In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import FuncFormatter, MaxNLocator
import seaborn as sns
from src.features import build_features

In [ ]:
plt.rcParams.update({'font.size': 14, 'font.family': 'serif'})

In [ ]:
data1 = pd.read_pickle(f"data/processed/Building01_Processed.pkl")
data2 = pd.read_pickle(f"data/processed/Building08_Processed.pkl")
data3 = pd.read_pickle(f"data/processed/Building11_Processed.pkl")

In [ ]:
features = [
    "Air_temperature",
    "Relative_humidity",
    "Surface_temperature",
    "Gas_volume",
    "Outside_air_temperature",
    "Outside_relative_humidity",
    "Wind_speed",
    "Total_rainfall",
]

feature_columns1 = [
        col for col in data1.columns if any(word in col for word in features)
    ]

feature_columns2 = [
        col for col in data2.columns if any(word in col for word in features)
    ]

feature_columns3 = [
        col for col in data3.columns if any(word in col for word in features)
    ]

In [ ]:
target_columns_1 = [
        col
        for col in data1.columns
        if any(col.endswith(word) for word in ["Air_temperature", "Relative_humidity"])
    ]

target_columns_2 = [
        col
        for col in data2.columns
        if any(col.endswith(word) for word in ["Air_temperature", "Relative_humidity"])
    ]

target_columns_3 = [
        col
        for col in data3.columns
        if any(col.endswith(word) for word in ["Air_temperature", "Relative_humidity"])
    ]

In [ ]:
# Define the spaces of interest
spaces_of_interest = ["Space13", "Space6", "Space9", "Space15"]

# Filter the columns based on spaces and feature types
air_temp_columns = [col for col in feature_columns1 if "Air_temperature" in col and any(space in col for space in spaces_of_interest)]
humidity_columns = [col for col in feature_columns1 if "Relative_humidity" in col and any(space in col for space in spaces_of_interest)]
surface_temp_columns = [col for col in feature_columns1 if "Surface_temperature" in col and any(space in col for space in spaces_of_interest)]

# Assuming 'data' is your DataFrame and it has been filtered for January 2014
jan_2014_data = data1.loc['2014-01-01':'2014-01-31']

# Function to plot the data with the final graph settings
def plot_multiple_metrics(data, air_temp_columns, humidity_columns, surface_temp_columns, gas_volume_column):
    fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(20, 15))
    
    # General settings based on MATLAB preferences
    fontsize = 26
    lineWidth = 3
    borderwidth = 2
    marker_size = 6
    alpha_value = 0.9
    legend_fontsize = 18
    
    # Define color map for consistent colors across plots
    colors = ['b', 'r', 'g', 'm']
    
    # Plot Air Temperature
    for col, color in zip(air_temp_columns, colors):
        axes[0, 0].plot(data[col], label=col, linewidth=lineWidth, color=color, alpha=alpha_value, markersize=marker_size)
    axes[0, 0].set_ylabel("Air Temperature (\u00B0C)", fontsize=fontsize, color='k', labelpad=10)
    axes[0, 0].tick_params(axis='both', which='major', labelsize=fontsize)
    axes[0, 0].spines['top'].set_linewidth(borderwidth)
    axes[0, 0].spines['right'].set_linewidth(borderwidth)
    axes[0, 0].spines['bottom'].set_linewidth(borderwidth)
    axes[0, 0].spines['left'].set_linewidth(borderwidth)
    axes[0, 0].grid(True, axis='y')
    axes[0, 0].tick_params(labelbottom=False)

    # Plot Relative Humidity
    for col, color in zip(humidity_columns, colors):
        axes[0, 1].plot(data[col], label=col, linewidth=lineWidth, color=color, alpha=alpha_value, markersize=marker_size)
    axes[0, 1].set_ylabel("Relative Humidity (%)", fontsize=fontsize, color='k', labelpad=10)
    axes[0, 1].tick_params(axis='both', which='major', labelsize=fontsize)
    axes[0, 1].spines['top'].set_linewidth(borderwidth)
    axes[0, 1].spines['right'].set_linewidth(borderwidth)
    axes[0, 1].spines['bottom'].set_linewidth(borderwidth)
    axes[0, 1].spines['left'].set_linewidth(borderwidth)
    axes[0, 1].grid(True, axis='y')
    axes[0, 1].tick_params(labelbottom=False)

    # Plot Surface Temperature
    for col, color in zip(surface_temp_columns, colors):
        axes[1, 0].plot(data[col], label=col, linewidth=lineWidth, color=color, alpha=alpha_value, markersize=marker_size)
    axes[1, 0].set_xlabel("Day", fontsize=fontsize, color='k')
    axes[1, 0].set_ylabel("Surface Temperature (\u00B0C)", fontsize=fontsize, color='k', labelpad=10)
    axes[1, 0].tick_params(axis='both', which='major', labelsize=fontsize)
    axes[1, 0].spines['top'].set_linewidth(borderwidth)
    axes[1, 0].spines['right'].set_linewidth(borderwidth)
    axes[1, 0].spines['bottom'].set_linewidth(borderwidth)
    axes[1, 0].spines['left'].set_linewidth(borderwidth)
    axes[1, 0].grid(True, axis='y')

    # Plot Gas Volume
    axes[1, 1].plot(data[gas_volume_column], label=gas_volume_column, linewidth=lineWidth, color='#66c2a5', alpha=alpha_value, markersize=marker_size)
    axes[1, 1].set_xlabel("Day", fontsize=fontsize, color='k')
    axes[1, 1].set_ylabel("Gas Volume ($\mathrm{m^3}$)", fontsize=fontsize, color='k', labelpad=10)
    axes[1, 1].tick_params(axis='both', which='major', labelsize=fontsize)
    axes[1, 1].spines['top'].set_linewidth(borderwidth)
    axes[1, 1].spines['right'].set_linewidth(borderwidth)
    axes[1, 1].spines['bottom'].set_linewidth(borderwidth)
    axes[1, 1].spines['left'].set_linewidth(borderwidth)
    axes[1, 1].grid(True, axis='y')

    # Format x-axis to show month/year at the start and day elsewhere
    def custom_date_formatter(x, pos):
        date = mdates.num2date(x)
        if pos == 0:
            return date.strftime('%d\n\n%b %Y')  # Day + Month Year at the start
        else:
            return date.strftime('%d')  # Display only the day

    day_formatter = FuncFormatter(custom_date_formatter)
    for ax in axes[1, :]:
        ax.xaxis.set_major_formatter(day_formatter)
    
    # Add a single legend for all subplots
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, spaces_of_interest, loc='upper center', ncol=4, fontsize=legend_fontsize, title='Building A Spaces')
    
    # plt.gcf().autofmt_xdate()

    plt.tight_layout(rect=[0, 0, 1, 0.93])  # Adjust layout to make space for the legend
    
    plt.savefig('reports/figures/buildingA_data.pdf', dpi=300, format='pdf')
    
    plt.show()
    plt.close()

# Define the gas volume column
gas_volume_column = "Gas_volume"

# Plot the metrics for the filtered data and columns
plot_multiple_metrics(jan_2014_data, air_temp_columns, humidity_columns, surface_temp_columns, gas_volume_column)

In [ ]:
def two_line_subplot(data):
    """Create subplots for Gas volume with Air temperature and Relative humidity."""
    fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(20, 15))

    # General settings based on previous plots
    fontsize = 26
    lineWidth = 3
    borderwidth = 2
    marker_size = 6
    alpha_value = 0.9
    legend_fontsize = 18

    # Define colors
    color1 = "b"  # Blue
    color2 = "r"  # Orange

    # Custom date formatter to show month/year at the start and days after
    def custom_date_formatter(x, pos):
        date = mdates.num2date(x)
        if pos == 0:
            return date.strftime('%d\n\n%b %Y')  # Show day and month/year on the first tick
        else:
            return date.strftime('%d')  # Show only the day on subsequent ticks

    day_formatter = FuncFormatter(custom_date_formatter)

    # Plot Gas volume with Air temperature
    ax1 = axes[0]
    col1 = "Gas_volume"
    col2 = "Space13_Air_temperature"
    ax1.plot(data.index, data[col1], label=col1, color=color1, linewidth=lineWidth, alpha=alpha_value)
    ax1.set_ylabel("Gas Volume (m³)", fontsize=fontsize, color='k', labelpad=10)
    ax1.tick_params(axis="y", labelcolor='k', labelsize=fontsize)
    ax1.spines['top'].set_linewidth(borderwidth)
    ax1.spines['right'].set_linewidth(borderwidth)
    ax1.spines['bottom'].set_linewidth(borderwidth)
    ax1.spines['left'].set_linewidth(borderwidth)

    ax3 = ax1.twinx()
    ax3.plot(data.index, data[col2], label=col2, color=color2, linestyle="--", linewidth=lineWidth)
    ax3.set_ylabel("Air Temperature (°C)", fontsize=fontsize, color='k', labelpad=10)
    ax3.tick_params(axis="y", labelcolor='k', labelsize=fontsize)
    ax3.spines['top'].set_linewidth(borderwidth)
    ax3.spines['right'].set_linewidth(borderwidth)
    ax3.spines['bottom'].set_linewidth(borderwidth)
    ax3.spines['left'].set_linewidth(borderwidth)

    ax1.grid(True)  # Show only horizontal grid lines
    ax1.tick_params(labelbottom=False)

    ax1.legend(loc="upper left", fontsize=legend_fontsize)
    ax3.legend(loc="upper right", fontsize=legend_fontsize)

    # Plot Gas volume with Relative humidity
    ax2 = axes[1]
    col2 = "Space13_Relative_humidity"
    ax2.plot(data.index, data[col1], label=col1, color=color1, linewidth=lineWidth, alpha=alpha_value)
    ax2.set_ylabel("Gas Volume (m³)", fontsize=fontsize, color='k', labelpad=10)
    ax2.tick_params(axis="y", labelcolor='k', labelsize=fontsize)
    ax2.set_xlabel("Day", fontsize=fontsize)
    ax2.spines['top'].set_linewidth(borderwidth)
    ax2.spines['right'].set_linewidth(borderwidth)
    ax2.spines['bottom'].set_linewidth(borderwidth)
    ax2.spines['left'].set_linewidth(borderwidth)

    ax4 = ax2.twinx()
    ax4.plot(data.index, data[col2], label=col2, color=color2, linestyle="--", linewidth=lineWidth)
    ax4.set_ylabel("Relative Humidity (%)", fontsize=fontsize, color='k', labelpad=10)
    ax4.tick_params(axis="y", labelcolor='k', labelsize=fontsize)
    ax4.spines['top'].set_linewidth(borderwidth)
    ax4.spines['right'].set_linewidth(borderwidth)
    ax4.spines['bottom'].set_linewidth(borderwidth)
    ax4.spines['left'].set_linewidth(borderwidth)

    # ax2.xaxis.set_major_locator(MaxNLocator(nbins=5))
    ticks = [mdates.date2num(data.index[0])] + list(ax2.get_xticks())
    ax2.set_xticks(ticks)
    ax2.xaxis.set_major_formatter(day_formatter)

    ax2.grid(True)  # Show only horizontal grid lines

    ax2.legend(loc="upper left", fontsize=legend_fontsize)
    ax4.legend(loc="upper right", fontsize=legend_fontsize)

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(f'reports/figures/buildingA_two_line_data.pdf', dpi=300, format='pdf')
    plt.show()
    plt.close()

# Example usage
jan_2014_data = data1.loc['2014-01-01':'2014-01-31']
two_line_subplot(jan_2014_data)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import FuncFormatter, MaxNLocator

def day_one_subplot(data, building_id):
    """Create subplots for Gas volume with Air temperature and Surface temperature on Day 1."""
    fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(20, 15))

    # General settings
    fontsize = 26
    lineWidth = 3
    borderwidth = 2
    marker_size = 6
    alpha_value = 0.9
    legend_fontsize = 18

    # Define colors
    color1 = "b"  # Blue
    color2 = "r"  # Orange

    # Custom date formatter to show hours only
    def custom_time_formatter(x, pos):
        date = mdates.num2date(x)
        if pos == 0:
            return date.strftime('%H:%M\n\n%d %b')  # Show day and month/year on the first tick
        else:
            return date.strftime('%H:%M')
        

    time_formatter = FuncFormatter(custom_time_formatter)

    # Plot Gas volume with Air temperature
    ax1 = axes[0]
    col1 = "Gas_volume"
    col2 = "Space13_Air_temperature"
    ax1.plot(data.index, data[col1], label=col1, color=color1, linewidth=lineWidth, alpha=alpha_value)
    ax1.set_ylabel("Gas Volume (m³)", fontsize=fontsize, color='k', labelpad=10)
    ax1.tick_params(axis="y", labelcolor='k', labelsize=fontsize)
    ax1.spines['top'].set_linewidth(borderwidth)
    ax1.spines['right'].set_linewidth(borderwidth)
    ax1.spines['bottom'].set_linewidth(borderwidth)
    ax1.spines['left'].set_linewidth(borderwidth)

    ax3 = ax1.twinx()
    ax3.plot(data.index, data[col2], label=col2, color=color2, linestyle="--", linewidth=lineWidth)
    ax3.set_ylabel("Air Temperature (°C)", fontsize=fontsize, color='k', labelpad=10)
    ax3.tick_params(axis="y", labelcolor='k', labelsize=fontsize)
    ax3.spines['top'].set_linewidth(borderwidth)
    ax3.spines['right'].set_linewidth(borderwidth)
    ax3.spines['bottom'].set_linewidth(borderwidth)
    ax3.spines['left'].set_linewidth(borderwidth)

    ax1.grid(True)  # Show only horizontal grid lines
    ax1.tick_params(labelbottom=False)

    ax1.legend(loc="upper left", fontsize=legend_fontsize)
    ax3.legend(loc="upper right", fontsize=legend_fontsize)

    # Plot Gas volume with Surface temperature
    ax2 = axes[1]
    col2 = "Space13_Surface_temperature"
    ax2.plot(data.index, data[col1], label=col1, color=color1, linewidth=lineWidth, alpha=alpha_value)
    ax2.set_ylabel("Gas Volume (m³)", fontsize=fontsize, color='k', labelpad=10)
    ax2.tick_params(axis="y", labelcolor='k', labelsize=fontsize)
    ax2.set_xlabel("Time", fontsize=fontsize)
    ax2.spines['top'].set_linewidth(borderwidth)
    ax2.spines['right'].set_linewidth(borderwidth)
    ax2.spines['bottom'].set_linewidth(borderwidth)
    ax2.spines['left'].set_linewidth(borderwidth)

    ax4 = ax2.twinx()
    ax4.plot(data.index, data[col2], label=col2, color=color2, linestyle="--", linewidth=lineWidth)
    ax4.set_ylabel("Surface Temperature (°C)", fontsize=fontsize, color='k', labelpad=10)
    ax4.tick_params(axis="y", labelcolor='k', labelsize=fontsize)
    ax4.spines['top'].set_linewidth(borderwidth)
    ax4.spines['right'].set_linewidth(borderwidth)
    ax4.spines['bottom'].set_linewidth(borderwidth)
    ax4.spines['left'].set_linewidth(borderwidth)

    ax2.xaxis.set_major_formatter(time_formatter)

    ax2.grid(True)  # Show only horizontal grid lines

    ax2.legend(loc="upper left", fontsize=legend_fontsize)
    ax4.legend(loc="upper right", fontsize=legend_fontsize)

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(f'reports/figures/buildingA_day1_data.pdf', dpi=300, format='pdf')
    plt.show()
    plt.close()

# Example usage for Day 1
day1_data = data1.loc['2014-01-16']
day_one_subplot(day1_data, "Building1")

In [ ]:
import matplotlib.pyplot as plt
import statsmodels.api as sm

def plot_autocorr_and_pacf_combined(data, columns, building_id, lags=40):
    """Plot autocorrelation on the left and partial autocorrelation on the right for each column."""
    fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(20, 15))

    # Final graph settings
    fontsize = 26
    lineWidth = 3
    borderwidth = 2
    blue_color = 'b'  # Define color for both lines and markers

    for i, column in enumerate(columns):
        # Plot Autocorrelation (left)
        sm.graphics.tsa.plot_acf(
            data[column].dropna(),
            lags=lags,
            ax=axes[i, 0],  # Row i, Column 0
            vlines_kwargs={'colors': blue_color},
        )
        axes[i, 0].set_ylabel(f"{column.split('_')[1]} {column.split('_')[2]}", fontsize=fontsize, color='k', labelpad=10)
        axes[i, 0].tick_params(axis="y", labelcolor='k', labelsize=fontsize)  # Apply settings to y-axis only
        if i == 0:
            axes[i, 0].set_title(f"Autocorrelation", fontsize=fontsize)
            axes[i, 0].tick_params(labelbottom=False)  # Remove x-axis labels for the first row
        else:
            axes[i, 0].tick_params(axis="x", labelcolor='k', labelsize=fontsize)  # Apply settings to x-axis for the second row
            axes[i, 0].set_title('')
            
        axes[i, 0].spines['top'].set_linewidth(borderwidth)
        axes[i, 0].spines['right'].set_linewidth(borderwidth)
        axes[i, 0].spines['bottom'].set_linewidth(borderwidth)
        axes[i, 0].spines['left'].set_linewidth(borderwidth)
        axes[i, 0].grid(True)

        # Plot Partial Autocorrelation (right)
        sm.graphics.tsa.plot_pacf(
            data[column].dropna(),
            lags=lags,
            ax=axes[i, 1],  # Row i, Column 1
            vlines_kwargs={'colors': blue_color},
        )
        if i == 0:
            axes[i, 1].set_title(f"Partial Autocorrelation", fontsize=fontsize)
            axes[i, 1].tick_params(labelbottom=False)  # Remove x-axis labels for the first row
        else:
            axes[i, 1].tick_params(axis="x", labelcolor='k', labelsize=fontsize)  # Apply settings to x-axis for the second row
            axes[i, 1].set_title('')
        axes[i, 1].tick_params(labelleft=False)
        
        axes[i, 1].spines['top'].set_linewidth(borderwidth)
        axes[i, 1].spines['right'].set_linewidth(borderwidth)
        axes[i, 1].spines['bottom'].set_linewidth(borderwidth)
        axes[i, 1].spines['left'].set_linewidth(borderwidth)
        axes[i, 1].grid(True)

    # Adjust layout and save the plot
    plt.tight_layout(rect=[0, 0.12, 1, 1])
    plt.savefig(f"reports/figures/buildingB_combined_auto_par_correlation.pdf", dpi=300, format='pdf')
    plt.show()
    plt.close()

# Filter data for February 2014
feb_2014_data = data2.loc['2014-02-01':'2014-02-28']

# Columns to plot
columns_to_plot = ["Space124_Air_temperature", "Space124_Relative_humidity"]

# Plot the combined autocorrelation and partial autocorrelation
plot_autocorr_and_pacf_combined(feb_2014_data, columns_to_plot, "BuildingB")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

def plot_combined_binary_outliers(dataset, building_id, reset_index=False):
    """Create subplots for Gas volume and Wind speed with outliers highlighted"""
    if reset_index:
        dataset = dataset.reset_index()
    
    # General settings based on MATLAB preferences
    fontsize = 26
    lineWidth = 3
    borderwidth = 2
    marker_size = 6
    alpha_value = 0.9
    legend_fontsize = 18

    # Setting up figure and axes
    fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(20, 15))
    plt.rcParams.update({'font.size': fontsize, 'font.family': 'serif'})

    # Define colors
    data_color = 'b'  # Navy Blue
    outlier_color = "r"  # Bright Yellow
    
    # Plot Gas volume
    col1 = "Gas_volume"
    outlier_col1 = col1 + "_outlier"
    dataset_gv = dataset.dropna(subset=[col1, outlier_col1])
    dataset_gv[outlier_col1] = dataset_gv[outlier_col1].astype("bool")

    axes[0].plot(dataset_gv.index, dataset_gv[col1], marker="o", linestyle="", alpha=alpha_value, color=data_color, markersize=marker_size, linewidth=lineWidth)
    axes[0].plot(
        dataset_gv.loc[dataset_gv[outlier_col1], col1],
        marker="x",
        linestyle="",
        color=outlier_color,
        label=f"Outliers in {col1}",
        markersize=8,
        linewidth=lineWidth
    )
    # axes[0].set_xlabel("Date", fontsize=fontsize, color='k', labelpad=15)
    axes[0].set_ylabel("Gas Volume ($\mathrm{m^3}$)", fontsize=fontsize, color='k', labelpad=40)
    axes[0].tick_params(axis='both', which='major', labelsize=fontsize)
    legend = axes[0].legend(loc='upper right', frameon=True, fontsize=legend_fontsize)
    legend.get_frame().set_edgecolor('k')
    axes[0].spines['top'].set_linewidth(borderwidth)
    axes[0].spines['right'].set_linewidth(borderwidth)
    axes[0].spines['bottom'].set_linewidth(borderwidth)
    axes[0].spines['left'].set_linewidth(borderwidth)
    axes[0].grid(True)
    
    axes[0].tick_params(labelbottom=False)

    # Plot Wind speed
    col2 = "Wind_speed"
    outlier_col2 = col2 + "_outlier"
    dataset_ws = dataset.dropna(subset=[col2, outlier_col2])
    dataset_ws[outlier_col2] = dataset_ws[outlier_col2].astype("bool")

    axes[1].plot(dataset_ws.index, dataset_ws[col2], marker="o", linestyle="", alpha=alpha_value, color=data_color, markersize=marker_size, linewidth=lineWidth)
    axes[1].plot(
        dataset_ws.loc[dataset_ws[outlier_col2], col2],
        marker="x",
        linestyle="",
        color=outlier_color,
        label=f"Outliers in {col2}",
        markersize=8,
        linewidth=lineWidth
    )
    axes[1].set_xlabel("Date", fontsize=fontsize, color='k', labelpad=25)
    axes[1].set_ylabel("Wind Speed (m/s)", fontsize=fontsize, color='k', labelpad=10)
    axes[1].tick_params(axis='both', which='major', labelsize=fontsize)
    legend = axes[1].legend(loc='upper right', frameon=True, fontsize=legend_fontsize)
    legend.get_frame().set_edgecolor('k')
    axes[1].spines['top'].set_linewidth(borderwidth)
    axes[1].spines['right'].set_linewidth(borderwidth)
    axes[1].spines['bottom'].set_linewidth(borderwidth)
    axes[1].spines['left'].set_linewidth(borderwidth)
    axes[1].grid(True)
    
    plt.gcf().autofmt_xdate()
    
    # Improve layout and save as a high-resolution image
    plt.tight_layout(pad=3.0)
    plt.savefig(f"reports/figures/buildingA_outliers.pdf", dpi=300, format='pdf')
    plt.show()
    plt.close()


# Example usage remains the same
columns_to_plot = [
    "Gas_volume",
    "Wind_speed",
]
outlier_col_suffix = "_outlier"  # Suffix for the outlier columns
building_id = "Building01"
pre_data = pd.read_pickle(f"data/interim/{building_id}_Preprocessed.pkl")

pre_data = build_features.remove_sparse_columns(pre_data, threshold=0.2)
sub_data = build_features.data_slicing(pre_data, pre_data.columns)

# Apply your outlier detection method here
for col in columns_to_plot:
    sub_data = build_features.mark_outliers_chauvenet(sub_data, col)  # Example placeholder

plot_combined_binary_outliers(sub_data, building_id)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Prepare the data
data1_copy = data1.copy()
data1_copy["year"] = data1_copy.index.year
data1_copy["month"] = data1_copy.index.month
data1_copy["weekday"] = data1_copy.index.weekday
data1_copy["hour"] = data1_copy.index.hour

# Create pivot table
plot_df = pd.pivot_table(data1, index="weekday", values='Gas_volume', columns="hour", aggfunc="mean")

# Final graph settings
fontsize = 26
borderwidth = 2
cbar_fontsize = 26  # Set color bar label font size equal to other labels
title_fontsize = 26

# Create heatmap without space between cells
fig, ax = plt.subplots(figsize=(20, 15))
sns.heatmap(
    plot_df, 
    ax=ax, 
    cmap='plasma', 
    fmt=".1f", 
    linewidths=0,  # No space between cells by setting linewidths to 0
    cbar_kws={'label': 'Mean Gas Volume', 'shrink': 0.75, 'orientation': 'vertical'}  # Color bar settings
)

# Set x and y labels
ax.set_xlabel("Hour", fontsize=fontsize, labelpad=10)
ax.set_ylabel("Weekday", fontsize=fontsize, labelpad=10)

# Set tick parameters
ax.tick_params(axis='both', which='major', labelsize=fontsize)

# Customize the color bar
cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=cbar_fontsize)  # Tick labels on the color bar
cbar.set_label('Mean Gas Volume', fontsize=cbar_fontsize, labelpad=10)  # Set color bar label font size

# Customize the spines' appearance
for spine in ax.spines.values():
    spine.set_linewidth(borderwidth)

# Adjust layout to prevent label cutoff
plt.tight_layout(rect=[0, 0, 1, 0.98])

# Save the plot
plt.savefig(f"reports/figures/buildingA_gas_volume_weekday.pdf", dpi=300, format='pdf')

# Show the plot
plt.show()
plt.close()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Define model names and file paths for all three buildings
model_names = [
    "LinearRegression",
    "RidgeCV",
    "MultiTaskLassoCV",
    "DecisionTree",
    "RandomForest",
    "XGBoost",
    "MLP"
]

buildings = ["Building01", "Building08", "Building11"]

# Combine data from all buildings
all_data = []

for building in buildings:
    for model in model_names:
        file_path = f"reports/cv_models_metrics/{building}_{model}_metrics.csv"
        df = pd.read_csv(file_path)
        df["model"] = model
        df["building"] = building
        all_data.append(df)

# Concatenate all data
data = pd.concat(all_data, ignore_index=True)
data = data.sort_values(by=['building', 'RMSE'], ascending=True)

# Plot settings
fontsize = 26
lineWidth = 3
borderwidth = 2
legend_fontsize = 18

plt.figure(figsize=(20, 15))
sns.boxplot(x="building", y='RMSE', hue="model", data=data, palette="Set1")
plt.xlabel('Buildings', fontsize=fontsize, labelpad=25)
plt.ylabel('RMSE', fontsize=fontsize, labelpad=25)
plt.xticks(ticks=range(len(buildings)), labels=["BuildingA", "BuildingB", "BuildingC"], fontsize=fontsize)
plt.yticks(fontsize=fontsize)
plt.legend(title='Models', fontsize=legend_fontsize, title_fontsize=legend_fontsize)
plt.grid(True)

plt.tight_layout(rect=[0, 0, 1, 0.95])

# Save plot
plt.savefig('reports/figures/buildings_rmse_comparison.pdf', dpi=300, format="pdf")
plt.show()
plt.close()

In [ ]:
import json

def format_param_value(value):
    if isinstance(value, (float)):
        return f"{value:.4f}"
    return value

# Load JSON files
with open('models/models_best_params/Building01_best_params.json') as f1, open('models/models_best_params/Building08_best_params.json') as f2, open('models/models_best_params/Building11_best_params.json') as f3:
    data_01 = json.load(f1)
    data_08 = json.load(f2)
    data_11 = json.load(f3)

# List of buildings
buildings = ['A', 'B', 'C']

# Models used (adjust based on actual models)
models = set(data_01.keys()).union(data_08.keys()).union(data_11.keys())

# Prepare LaTeX table
table = r"""
\begin{table}[h!]
    \centering
    \caption{Optimal hyperparameters for buildings, detailing the configurations that yielded the best performance across 5 folds of the dataset}
    \label{tab:best_params}
    \begin{tabular}{llccc}
    \hline
    \textbf{Model} & \textbf{Best Hyperparameters} & \multicolumn{3}{c}{\textbf{Buildings}} \\ \cline{3-5} 
    & & \textbf{A} & \textbf{B} & \textbf{C} \\ \hline
"""

# Add models and parameters
for model in models:
    params_01 = data_01.get(model, {})
    params_08 = data_08.get(model, {})
    params_11 = data_11.get(model, {})

    # Get the list of all parameters used by the model
    all_params = set(params_01.keys()).union(params_08.keys()).union(params_11.keys())

    if not all_params:
        table += f"{model} & None & - & - & - \\\\\n"
    else:
        for i, param in enumerate(all_params):
            formatted_param_01 = format_param_value(params_01.get(param, '-'))
            formatted_param_08 = format_param_value(params_08.get(param, '-'))
            formatted_param_11 = format_param_value(params_11.get(param, '-'))
            
            param = param.replace("_", "\\_") 
            
            if i == 0:
                table += f"\\multirow{{{len(all_params)}}}{{*}}{{{model}}} & {param} & {formatted_param_01} & {formatted_param_08} & {formatted_param_11} \\\\\n"
            else:
                table += f"& {param} & {formatted_param_01} & {formatted_param_08} & {formatted_param_11} \\\\\n"
    table += "\\hline\n"

table += r"""
    \end{tabular}
\end{table}
"""

# Save to a .tex file
with open("models/models_best_params/best_hyperparameters_table.tex", "w") as tex_file:
    tex_file.write(table)

In [ ]:
import pandas as pd

# Define models and their corresponding file paths for each building
models = ["LinearRegression", "RidgeCV", "MultiTaskLassoCV", "DecisionTree", "RandomForest", "XGBoost", "MLP"]
buildings = ["Building 01", "Building 08", "Building 11"]

# Define a dictionary to hold file paths for metrics
file_paths = {
    "Building 01": {model: f"reports/predictions/Building01_{model}_metrics.csv" for model in models},
    "Building 08": {model: f"reports/predictions/Building08_{model}_metrics.csv" for model in models},
    "Building 11": {model: f"reports/predictions/Building11_{model}_metrics.csv" for model in models}
}

# Define file paths for computation time
compute_time_paths = {
    "Building 01": "reports/predictions/Building01_computation_time.csv",
    "Building 08": "reports/predictions/Building08_computation_time.csv",
    "Building 11": "reports/predictions/Building11_computation_time.csv"
}

building_mapping = {
    "Building 01": "Building A",
    "Building 08": "Building B",
    "Building 11": "Building C"
}

# Initialize a dictionary to store results
results = {building: {model: {} for model in models} for building in buildings}

# Process each building and model file
for building in buildings:
    for model in models:
        # Read the CSV file for the model and building
        df = pd.read_csv(file_paths[building][model])

        # Calculate mean and standard deviation for each metric
        metrics_mean = df.mean()
        metrics_std = df.std()

        # Store the formatted results
        results[building][model] = {
            "Time": None,
            "MAE": metrics_mean['MAE'],
            "RMSE": metrics_mean['RMSE'],
            "R2": metrics_mean['R2 Score']
        }

for building in buildings:
    compute_time_df = pd.read_csv(compute_time_paths[building])
    for model in models:
        compute_time = compute_time_df.loc[compute_time_df["Model"] == model]["Computation Time (s)"].values[0] / 5
        results[building][model]["Time"] = compute_time

# Generate LaTeX table
latex_table = r"""
\begin{table}[h!]
    \centering
    \caption{Model performance metrics and computation time for Buildings A, B, and C}
    \label{tab:buildings_cv_results}
    \begin{tabular}{llcccc}
    \hline
    \textbf{Building} & \textbf{Model} & \textbf{Time (s)} & \textbf{MAE} & \textbf{RMSE} & \textbf{R$^2$} \\ \hline
"""

# Add rows for each building and model with multirow for building name
for building in buildings:
    name = building_mapping[building]
    latex_table += f"\\multirow{{{len(models)}}}{{*}}{{{name}}}"
    for i, model in enumerate(models):
        row = f"& {model} & {results[building][model]['Time']:.2f} & {results[building][model]['MAE']:.4f} & {results[building][model]['RMSE']:.4f} & {results[building][model]['R2']:.4f} \\\\\n"
        latex_table += row
        
    latex_table += "\\hline\n"

# Close the table
latex_table += r"""
    \end{tabular}
\end{table}
"""

# Save to a .tex file
with open("reports/predictions/buildings_cv_results.tex", "w") as tex_file:
    tex_file.write(latex_table)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

def feature_importance_plot(feat_df, column, building_id):
    # Prepare final plot settings
    fontsize = 26
    borderwidth = 2
    
    # Prepare data
    sub_df = feat_df[[column]].copy()
    sub_df["_abs_imp"] = np.abs(sub_df[column])
    sub_df = sub_df.sort_values("_abs_imp", ascending=False).drop(columns="_abs_imp")
    
    # Plot
    fig, ax = plt.subplots(figsize=(20, 15))
    sns.barplot(x=sub_df.index, y=sub_df[column], ax=ax, palette="Set1")
    
    # Customize axes
    ax.set_xlabel('Features', fontsize=fontsize, labelpad=25)
    ax.set_ylabel('Importance', fontsize=fontsize, labelpad=10)
    ax.tick_params(axis='both', which='major', labelsize=fontsize)
    ax.tick_params(axis='x', rotation=90)  # Rotate x-ticks by 90 degrees
    
    # Set border width
    for spine in ax.spines.values():
        spine.set_linewidth(borderwidth)
    
    # Adjust layout and save figure
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.savefig(f'reports/figures/buildingC_feature_importance.pdf', dpi=300, format='pdf')
    plt.show()
    plt.close()

# Example usage
building_id = "Building11"
model_name = "linear_regression"
feat_df = pd.read_csv(f"models/feature_order/{building_id}_{model_name}_feature_importance.csv", index_col=0)

col = "Space169_Air_temperature"
feature_importance_plot(feat_df, col, building_id)

In [ ]:
import pandas as pd

# Define the mapping for prediction horizons
horizon_mapping = {
    0: "One-step (30 min)",
    1: "1-hour",
    3: "3-hour",
    6: "6-hour",
    9: "9-hour",
    12: "12-hour",
    24: "24-hour",
    48: "48-hour"
}

# File paths for each building
buildings = ["Building01", "Building08", "Building11"]
building_names = {
    "Building01": "Building A",
    "Building08": "Building B",
    "Building11": "Building C"
}

# Initialise a dictionary to hold data
results = {building: pd.DataFrame() for building in buildings}

# Read data for each building
for building in buildings:
    file_path = f"reports/multistep_predictions/{building}_linear_regression_metrics.csv"
    df = pd.read_csv(file_path)
    df['Building'] = building_names[building]
    df['Prediction Horizon'] = df['Prediction Horizon'].astype(int).map(horizon_mapping)
    results[building] = df

# Combine all building data into one DataFrame
all_data = pd.concat(results.values(), ignore_index=True)

# Generate LaTeX table
latex_table = r"""
\begin{table}[h!]
    \centering
    \caption{Performance metrics of the linear regression model for different prediction horizons for Buildings A, B, and C.}
    \label{tab:prediction_horizons}
    \begin{tabular}{lcccc}
    \hline
    \textbf{Building} & \textbf{Prediction Horizon} & \textbf{MAE} & \textbf{RMSE} & \textbf{R$^2$ Score} \\ \hline
"""

# Add rows for each building and prediction horizon
for building in building_names.values():
    building_df = all_data[all_data['Building'] == building]
    latex_table += f"\\multirow{{{len(building_df)}}}{{*}}{{{building}}}"
    for i, row in building_df.iterrows():
        latex_table += f"& {row['Prediction Horizon']} & {row['MAE']:.4f} & {row['RMSE']:.4f} & {row['R2 Score']:.4f} \\\\\n"
    latex_table += "\\hline\n"

# Close the table
latex_table += r"""
    \end{tabular}
\end{table}
"""

# Save to a .tex file
with open("reports/multistep_predictions/prediction_horizons_results.tex", "w") as tex_file:
    tex_file.write(latex_table)

In [ ]:
test_start_date = '2014-12-01'
test_end_date = '2015-03-31'

In [ ]:
y1 = data1[target_columns_1]
y_test1 = y1[(y1.index >= test_start_date) & (y1.index <= test_end_date)]

y2 = data2[target_columns_2]
y_test2 = y2[(y2.index >= test_start_date) & (y2.index <= test_end_date)]

y3 = data3[target_columns_3]
y_test3 = y3[(y3.index >= test_start_date) & (y3.index <= test_end_date)]

In [ ]:
print(y_test1.shape)
print(y_test1.columns)

print(y_test2.shape)
print(y_test2.columns)

print(y_test3.shape)
print(y_test3.columns)

In [ ]:
import matplotlib.pyplot as plt

def plot_actual_vs_predicted(y_test, y_pred, building_id):
    """Plot actual vs predicted for Air temperature and Relative humidity."""
    fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(20, 15))

    # Plot settings
    fontsize = 26
    lineWidth = 3
    borderwidth = 2
    alpha_value = 0.9
    legend_fontsize = 18

    # Plot for Space169_Air_temperature
    col = "Space169_Air_temperature"
    ax1 = axes[0]
    ax1.plot(y_test.index, y_test[col], label="Actual", color="b", linewidth=lineWidth, alpha=alpha_value)
    ax1.plot(y_pred.index, y_pred[col], label="Predicted", linestyle="--", color="r", linewidth=lineWidth, alpha=alpha_value)
    ax1.set_ylabel(f"Air Temperature (°C)", fontsize=fontsize, color='k', labelpad=10)
    ax1.tick_params(axis="y", labelcolor='k', labelsize=fontsize)
    ax1.spines['top'].set_linewidth(borderwidth)
    ax1.spines['right'].set_linewidth(borderwidth)
    ax1.spines['bottom'].set_linewidth(borderwidth)
    ax1.spines['left'].set_linewidth(borderwidth)
    ax1.legend(loc="upper right", fontsize=legend_fontsize)
    ax1.grid(True)
    ax1.tick_params(labelbottom=False)

    # Plot for Space169_Relative_humidity
    col = "Space169_Relative_humidity"
    ax2 = axes[1]
    ax2.plot(y_test.index, y_test[col], label="Actual", color="b", linewidth=lineWidth, alpha=alpha_value)
    ax2.plot(y_pred.index, y_pred[col], label="Predicted", linestyle="--", color="r", linewidth=lineWidth, alpha=alpha_value)
    ax2.set_ylabel(f"Relative Humidity (%)", fontsize=fontsize, color='k', labelpad=10)
    ax2.set_xlabel("Date", fontsize=fontsize, labelpad=25)
    ax2.tick_params(axis="y", labelcolor='k', labelsize=fontsize)
    ax2.tick_params(axis='x', labelcolor='k', labelsize=fontsize)
    ax2.spines['top'].set_linewidth(borderwidth)
    ax2.spines['right'].set_linewidth(borderwidth)
    ax2.spines['bottom'].set_linewidth(borderwidth)
    ax2.spines['left'].set_linewidth(borderwidth)
    ax2.legend(loc="upper right", fontsize=legend_fontsize)
    ax2.grid(True)
    
    plt.gcf().autofmt_xdate()

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(f'reports/figures/{building_id}_one_step.pdf', dpi=300, format='pdf')
    plt.show()
    plt.close()

y_pred3_0 = pd.read_csv('reports/multistep_predictions/Building11_linear_regression_predictions_0h.csv')
y_pred3_0.index = y_test3.index
plot_actual_vs_predicted(y_test3, y_pred3_0, "buildingC")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import FuncFormatter

def plot_air_temp_predictions(y_test, y_pred):
    """Create a 2x2 subplot for actual vs predicted air temperature for four spaces."""
    fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(20, 15))

    # General plot settings
    fontsize = 26
    lineWidth = 3
    borderwidth = 2
    legend_fontsize = 18
    colors = ['b', 'r']
    spaces = ["Space6_Air_temperature", "Space9_Air_temperature", "Space13_Air_temperature", "Space15_Air_temperature"]

    # Custom date formatter
    def custom_date_formatter(x, pos):
        date = mdates.num2date(x)
        if pos == 0:
            return date.strftime('%d\n\n%b %Y')
        else:
            return date.strftime('%d')

    day_formatter = FuncFormatter(custom_date_formatter)

    # First subplot for Space 6
    ax = axes[0, 0]
    line1, = ax.plot(y_test.index, y_test[spaces[0]], label="Actual", color=colors[0], linewidth=lineWidth)
    line2, = ax.plot(y_test.index, y_pred[spaces[0]], label="Predicted", linestyle="--", color=colors[1], linewidth=lineWidth)
    ax.set_ylabel("Space 6", fontsize=fontsize, labelpad=10)
    ax.tick_params(axis='both', which='major', labelsize=fontsize)
    ax.xaxis.set_major_formatter(day_formatter)
    ax.spines.values()
    ax.grid(True)
    ax.tick_params(labelbottom=False)

    # Second subplot for Space 9
    ax = axes[0, 1]
    ax.plot(y_test.index, y_test[spaces[1]], label="Actual", color=colors[0], linewidth=lineWidth)
    ax.plot(y_test.index, y_pred[spaces[1]], label="Predicted", linestyle="--", color=colors[1], linewidth=lineWidth)
    ax.set_ylabel("Space 9", fontsize=fontsize, labelpad=10)
    ax.tick_params(axis='both', which='major', labelsize=fontsize)
    ax.xaxis.set_major_formatter(day_formatter)
    ax.grid(True)
    ax.tick_params(labelbottom=False)

    # Third subplot for Space 13
    ax = axes[1, 0]
    ax.plot(y_test.index, y_test[spaces[2]], label="Actual", color=colors[0], linewidth=lineWidth)
    ax.plot(y_test.index, y_pred[spaces[2]], label="Predicted", linestyle="--", color=colors[1], linewidth=lineWidth)
    ax.set_ylabel("Space 13", fontsize=fontsize, labelpad=10)
    ax.set_xlabel("Day", fontsize=fontsize)
    ax.tick_params(axis='both', which='major', labelsize=fontsize)
    ax.xaxis.set_major_formatter(day_formatter)
    ax.grid(True)

    # Fourth subplot for Space 15
    ax = axes[1, 1]
    ax.plot(y_test.index, y_test[spaces[3]], label="Actual", color=colors[0], linewidth=lineWidth)
    ax.plot(y_test.index, y_pred[spaces[3]], label="Predicted", linestyle="--", color=colors[1], linewidth=lineWidth)
    ax.set_ylabel("Space 15", fontsize=fontsize, labelpad=10)
    ax.set_xlabel("Day", fontsize=fontsize)
    ax.tick_params(axis='both', which='major', labelsize=fontsize)
    ax.xaxis.set_major_formatter(day_formatter)
    ax.grid(True)

    # Add a single legend for all subplots
    handles = [line1, line2]
    labels = ["Actual", "Predicted"]
    fig.legend(handles, labels, loc='upper center', ncol=4, fontsize=legend_fontsize, title='Building A Air Temperature', title_fontsize=fontsize)

    # Adjust layout and save figure
    plt.tight_layout(rect=[0, 0, 1, 0.93])
    plt.savefig(f'reports/figures/buildingA_air_temp_predictions.pdf', dpi=300, format='pdf')
    plt.show()
    plt.close()
    
# Example usage
y_pred1_0 = pd.read_csv('reports/multistep_predictions/Building01_linear_regression_predictions_0h.csv')
y_pred1_0.index = y_test1.index
plot_air_temp_predictions(y_test1.loc['2015-01-01':'2015-01-31'], y_pred1_0.loc['2015-01-01':'2015-01-31'])

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import FuncFormatter

def plot_air_temp_predictions(y_test, y_pred):
    """Create a 2x2 subplot for actual vs predicted air temperature for four spaces."""
    fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(20, 18))

    # General plot settings
    fontsize = 26
    lineWidth = 3
    borderwidth = 2
    legend_fontsize = 18
    colors = ['b', 'r']
    spaces = ["Space6_Relative_humidity", "Space9_Relative_humidity", "Space13_Relative_humidity", "Space15_Relative_humidity"]

    # Custom date formatter
    def custom_date_formatter(x, pos):
        date = mdates.num2date(x)
        if pos == 0:
            return date.strftime('%d\n\n%b %Y')
        else:
            return date.strftime('%d')

    day_formatter = FuncFormatter(custom_date_formatter)

    # First subplot for Space 6
    ax = axes[0, 0]
    line1, = ax.plot(y_test.index, y_test[spaces[0]], label="Actual", color=colors[0], linewidth=lineWidth)
    line2, = ax.plot(y_test.index, y_pred[spaces[0]], label="Predicted", linestyle="--", color=colors[1], linewidth=lineWidth)
    ax.set_ylabel("Space 6", fontsize=fontsize, labelpad=10)
    ax.tick_params(axis='both', which='major', labelsize=fontsize)
    ax.xaxis.set_major_formatter(day_formatter)
    ax.spines.values()
    ax.grid(True)
    ax.tick_params(labelbottom=False)

    # Second subplot for Space 9
    ax = axes[0, 1]
    ax.plot(y_test.index, y_test[spaces[1]], label="Actual", color=colors[0], linewidth=lineWidth)
    ax.plot(y_test.index, y_pred[spaces[1]], label="Predicted", linestyle="--", color=colors[1], linewidth=lineWidth)
    ax.set_ylabel("Space 9", fontsize=fontsize, labelpad=35)
    ax.tick_params(axis='both', which='major', labelsize=fontsize)
    ax.xaxis.set_major_formatter(day_formatter)
    ax.grid(True)
    ax.tick_params(labelbottom=False)

    # Third subplot for Space 13
    ax = axes[1, 0]
    ax.plot(y_test.index, y_test[spaces[2]], label="Actual", color=colors[0], linewidth=lineWidth)
    ax.plot(y_test.index, y_pred[spaces[2]], label="Predicted", linestyle="--", color=colors[1], linewidth=lineWidth)
    ax.set_ylabel("Space 13", fontsize=fontsize, labelpad=10)
    ax.set_xlabel("Day", fontsize=fontsize)
    ax.tick_params(axis='both', which='major', labelsize=fontsize)
    ax.xaxis.set_major_formatter(day_formatter)
    ax.grid(True)

    # Fourth subplot for Space 15
    ax = axes[1, 1]
    ax.plot(y_test.index, y_test[spaces[3]], label="Actual", color=colors[0], linewidth=lineWidth)
    ax.plot(y_test.index, y_pred[spaces[3]], label="Predicted", linestyle="--", color=colors[1], linewidth=lineWidth)
    ax.set_ylabel("Space 15", fontsize=fontsize, labelpad=10)
    ax.set_xlabel("Day", fontsize=fontsize)
    ax.tick_params(axis='both', which='major', labelsize=fontsize)
    ax.xaxis.set_major_formatter(day_formatter)
    ax.grid(True)

    # Add a single legend for all subplots
    handles = [line1, line2]
    labels = ["Actual", "Predicted"]
    fig.legend(handles, labels, loc='upper center', ncol=4, fontsize=legend_fontsize, title='Building A Relative Humidity', title_fontsize=fontsize)

    # Adjust layout and save figure
    plt.tight_layout(rect=[0, 0, 1, 0.93])
    plt.savefig(f'reports/figures/buildingA_relative_humi_predictions.pdf', dpi=300, format='pdf')
    plt.show()
    plt.close()
    
# Example usage
y_pred1_0 = pd.read_csv('reports/multistep_predictions/Building01_linear_regression_predictions_0h.csv')
y_pred1_0.index = y_test1.index
plot_air_temp_predictions(y_test1.loc['2015-01-01':'2015-01-31'], y_pred1_0.loc['2015-01-01':'2015-01-31'])

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import FuncFormatter

y_pred2_0h = pd.read_csv('reports/multistep_predictions/Building08_linear_regression_predictions_0h.csv')
y_pred2_6h = pd.read_csv('reports/multistep_predictions/Building08_linear_regression_predictions_6h.csv')
y_pred2_12h = pd.read_csv('reports/multistep_predictions/Building08_linear_regression_predictions_12h.csv')
y_pred2_24h = pd.read_csv('reports/multistep_predictions/Building08_linear_regression_predictions_24h.csv')

# Setting the index for predictions
y_pred2_0h.index = y_test2.index
y_pred2_6h.index = y_test2.index
y_pred2_12h.index = y_test2.index
y_pred2_24h.index = y_test2.index

date = '2015-01-17'
space = 'Space137_Air_temperature'

# Custom date formatter to show hours with date
def custom_time_formatter(x, pos):
    date = mdates.num2date(x)
    if pos == 0:
        return date.strftime('%H:%M\n\n%d %b')  # Show hour and day/month on the first tick
    else:
        return date.strftime('%H:%M')  # Show only the hour

time_formatter = FuncFormatter(custom_time_formatter)

# Plot settings
plt.figure(figsize=(20, 15))
fontsize = 26
lineWidth = 3
legend_fontsize = 18

# Plot the actual values
plt.plot(y_test2.loc[date].index, y_test2.loc[date][space], label='Actual', color='black', linewidth=lineWidth)

# Plot the predictions for each horizon
plt.plot(y_pred2_0h.loc[date].index, y_pred2_0h.loc[date][space], label='One-step (30 min)', linestyle='--', color='b', linewidth=lineWidth)
plt.plot(y_pred2_6h.loc[date].index, y_pred2_6h.loc[date][space], label='6-hour', linestyle='--', color='r', linewidth=lineWidth)
plt.plot(y_pred2_12h.loc[date].index, y_pred2_12h.loc[date][space], label='12-hour', linestyle='--', color='g', linewidth=lineWidth)
plt.plot(y_pred2_24h.loc[date].index, y_pred2_24h.loc[date][space], label='24-hour', linestyle='--', color='m', linewidth=lineWidth)

# Apply x-axis formatting
plt.gca().xaxis.set_major_formatter(time_formatter)

# Formatting the plot
plt.xlabel('Time', fontsize=fontsize)
plt.ylabel('Air Temperature (°C)', fontsize=fontsize, labelpad=10)
plt.legend(fontsize=legend_fontsize)
plt.xticks(fontsize=fontsize)
plt.yticks(fontsize=fontsize)
plt.grid(True)

# Save and show plot
plt.tight_layout()
plt.savefig('reports/figures/buildingB_pred_horizons.pdf', dpi=300, format="pdf")
plt.show()